# Tool catalog: camera, microphone, and screen

Run this notebook in a disposable JupyterLab project on a secure origin with a live Python kernel and selected ChatGPT connection. These are **real** camera, microphone, and screen operations, not synthetic device fixtures. Point the camera at a neutral wall, keep private windows outside the chosen screen area, and speak or make a brief sound for the microphone level. Grant browser permission only when its visible prompt appears. Each call returns a receipt quickly; wait for the media row to finish, then inspect it in the later cell. Recordings here run for at least five seconds. The final AI questions repeat bounded workflows through actual ChatGPT tool calls.


In [1]:
from nbinlineai.tools import (
    list_media_sources, start_camera, start_microphone, capture_camera, stop_source,
    start_recording, pause_recording, resume_recording, stop_recording,
    record_camera, record_microphone, read_audio_levels,
    setup_share, start_share, capture_screen, capture_tool, stop_share,
    operation_status, release_media,
)
from PIL import Image as PILImage
from base64 import b64encode
from html import escape
from IPython.display import display, HTML

def show_media_receipt(name, receipt):
    print(name, receipt.status, receipt.error)
    if receipt.status != 'completed':
        return
    result = receipt.result
    if isinstance(result, PILImage.Image):
        print('Captured image:', type(result).__name__, result.size)
        display(result.copy().resize((min(result.width, 320), min(result.height, 240))))
    elif hasattr(result, 'data') and hasattr(result, 'mime_type'):
        print('Recorded clip:', type(result).__name__, result.mime_type, len(result.data), 'bytes', getattr(result, 'duration_seconds', None), 'seconds')
        tag = 'audio' if result.mime_type.startswith('audio/') else 'video'
        if tag in ('audio', 'video') and result.mime_type.startswith(('audio/', 'video/')):
            uri = f"data:{escape(result.mime_type, quote=True)};base64,{b64encode(result.data).decode('ascii')}"
            display(HTML(f'<{tag} controls preload="metadata" src="{uri}"></{tag}>'))
    elif isinstance(result, dict) and 'devices' in result:
        print('Available devices:', len(result['devices']), [item.get('kind') for item in result['devices']])
    elif isinstance(result, dict):
        print('Result:', {key: value for key, value in result.items() if key not in ('device_id', 'label')})
    else:
        print('Result:', result)

# Camera-only steps may be unavailable while microphone and screen work.
camera_id = None
still = None
camera_clip = None


## list_media_sources

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [2]:
devices = list_media_sources()

Real hardware source discovery completed; labels and identifiers withheld from public copy.


In [3]:
show_media_receipt('list_media_sources', devices)


Real hardware source discovery completed; labels and identifiers withheld from public copy.


## start_camera

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [ ]:
camera = start_camera(audio=False)

In [ ]:
show_media_receipt('start_camera', camera)


## capture_camera

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [ ]:
camera_id = camera.result['source_id'] if camera.status == 'completed' else None
still = capture_camera(camera_id, save_to=None) if camera_id else None


In [ ]:
if still is not None:
    show_media_receipt('capture_camera', still)
else:
    print('Camera capture needs a completed camera source; continue with the microphone examples.')


## start_microphone

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [4]:
microphone = start_microphone()

In [5]:
show_media_receipt('start_microphone', microphone)


start_microphone completed None
Result: {'source_id': '[opaque hardware reference]', 'kind': 'microphone', 'audio': True, 'video': False}


## read_audio_levels

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [6]:
microphone_id = microphone.result["source_id"]
levels = read_audio_levels(microphone_id)

In [7]:
show_media_receipt('read_audio_levels', levels)


read_audio_levels completed None
Result: {'rms': 0.006, 'peak': 0.008, 'window_ms': 250}


## start_recording

Record from the live microphone source for at least three active seconds before pausing. Resume for at least three more active seconds before stopping. The 60-second wall-clock ceiling continues while paused; the resulting real audio clip should have at least five playable seconds.


In [8]:
recording = start_recording(microphone_id, save_to=None, duration=60)


In [9]:
show_media_receipt('start_recording', recording)


start_recording running None


## pause_recording

Pause after at least three seconds of active microphone recording. Paused time produces no playable audio, but the 60-second wall-clock ceiling still applies.


In [10]:
paused = pause_recording(recording.operation_id)

In [11]:
show_media_receipt('pause_recording', paused)


pause_recording completed None
Result: {'target_operation_id': 'AcMIg9sF3PEWvJccJQTQ2UJiXg4nJCGR', 'paused': True, 'tracks_remain_live': True}


## resume_recording

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [12]:
resumed = resume_recording(recording.operation_id)

In [13]:
show_media_receipt('resume_recording', resumed)


resume_recording completed None
Result: {'target_operation_id': 'AcMIg9sF3PEWvJccJQTQ2UJiXg4nJCGR', 'resumed': True}


## stop_recording

Wait at least three active seconds after resuming before stopping; combined active recording should be at least six seconds. Inspect the clip summary without printing its raw bytes.


In [14]:
stopped_recording = stop_recording(recording.operation_id)

In [15]:
show_media_receipt('stop_recording control', stopped_recording)
show_media_receipt('original recording after stop', recording)
if recording.status != 'completed':
    print('Wait for the original recording receipt to complete, then rerun this inspection cell.')


stop_recording control completed None
Result: {'target_operation_id': 'AcMIg9sF3PEWvJccJQTQ2UJiXg4nJCGR', 'target_status': 'completed'}
original recording after stop completed None
Recorded clip: MediaClip audio/webm;codecs=opus 332785 bytes 24.629099999904632 seconds


## stop_source

Stop the microphone source after the recording finishes. Inspect this control receipt on a later kernel turn; it does not remove the completed audio clip.


In [16]:
stopped_microphone = stop_source(microphone_id)


In [17]:
show_media_receipt('stop_source', stopped_microphone)


stop_source completed None
Result: {'source_id': '[opaque hardware reference]', 'stopped': True}


## record_camera

Record the neutral wall for five seconds. Allow camera permission in the browser if prompted, then inspect the actual clip type, byte count, and duration.


In [ ]:
camera_clip = record_camera(save_to=None, duration=5, audio=False)

In [ ]:
show_media_receipt('record_camera', camera_clip)


## record_microphone

Request seven seconds of real microphone audio so the decoded clip comfortably exceeds five seconds. A quiet voice or brief sound helps make the level reading meaningful. Grant permission in the browser if prompted.


In [18]:
microphone_clip = record_microphone(save_to=None, duration=7)

In [19]:
show_media_receipt('record_microphone', microphone_clip)


record_microphone completed None
Recorded clip: MediaClip audio/webm;codecs=opus 112313 bytes 7 seconds


## setup_share

Check the browser's screen-sharing controls before sharing. Keep private windows out of view.


In [2]:
sharing_controls = setup_share()

In [3]:
show_media_receipt('setup_share', sharing_controls)


setup_share completed None
Result: {'share_controls': True, 'screen_available': True}


## start_share

Click the browser's visible Share chooser and select only a neutral tab or area of this disposable project. Screen capture never starts silently.


In [ ]:
sharing = start_share(audio=False)

In [ ]:
show_media_receipt('start_share', sharing)


## capture_screen

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [ ]:
screen_id = sharing.result["source_id"]
screenshot = capture_screen(source_id=screen_id, save_to=None)

In [ ]:
show_media_receipt('capture_screen', screenshot)


## capture_tool

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [ ]:
alias_frame = capture_tool(source_id=screen_id, save_to=None)

In [ ]:
show_media_receipt('capture_tool', alias_frame)


## stop_share

Run the inspection cell on a later kernel turn. If the receipt is still running or waiting for you, use the notebook control and inspect it again.

In [ ]:
stopped_share = stop_share(screen_id)

In [ ]:
show_media_receipt('stop_share', stopped_share)


The microphone and screen sources are stopped above. The next cleanup stops the wall camera if it was available and releases only managed media IDs created here. Delivered Pillow images and clips remain ordinary Python objects until discarded.


In [ ]:
camera_cleanup = stop_source(camera_id) if camera_id else None
released_media = {}
seen_media_ids = set()
for label, receipt in [
    ('wall still', still), ('microphone recording', recording),
    ('camera clip', camera_clip), ('microphone clip', microphone_clip),
    ('screen still', screenshot), ('screen alias', alias_frame),
]:
    descriptor = receipt.media if receipt is not None and receipt.status == 'completed' else None
    media_id = descriptor.get('media_id') if isinstance(descriptor, dict) else None
    if media_id and media_id not in seen_media_ids:
        seen_media_ids.add(media_id)
        released_media[label] = release_media(media_id)
print('Requested release for', len(released_media), 'managed captures.')


In [ ]:
if camera_cleanup is not None:
    show_media_receipt('camera cleanup', camera_cleanup)
for label, receipt in released_media.items():
    show_media_receipt(f'release {label}', receipt)


## Ask ChatGPT to use the live devices

Run these short questions one at a time with the selected ChatGPT connection. A browser action first returns an operation ID and a running or waiting state; after your permission or recording interval, the separate status question checks the terminal result before another action uses its source ID. Keep each saved answer so the next question can use the exact ID. If camera access is unavailable, continue with the independent microphone recording and screen-sharing sections. Grant permissions and choose the neutral screen area yourself.


Call &`list_media_sources` with `kind="all"` and `limit=10` to list devices passively. Report only the immediate operation ID and current state. Do not request camera permission, report device counts yet, or start a camera.


Use &`operation_status` on the preceding `list_media_sources` operation ID after its media row finishes. If completed, report the camera and microphone counts on the returned page, whether more pages remain, and any omitted count, without labels or device IDs. If it is still active, stop and ask me to retry this status question later.


Call &`start_camera` with `audio=False` for my neutral wall. Report only its immediate operation ID and current state. Ask me to grant the visible camera permission; do not claim an active source yet.


After I grant camera permission, use &`operation_status` on the preceding `start_camera` operation ID. Report its verified source ID only if the operation completed. If it remains waiting or running, stop and retry this status question later before capturing.


Using only the completed camera source ID from the preceding status answer, call &`capture_camera` with `save_to=None`. Report its immediate operation ID/state; do not claim pixels or dimensions yet.


Use &`operation_status` on the prior `capture_camera` operation ID after its media row completes. Report actual image dimensions, MIME type, and managed media ID if present, without pixels or personal device details. If still running, stop and ask me to retry later.


Use &`stop_source` on the exact camera source ID confirmed above. Report the immediate control operation ID/state, and do not claim a different source was stopped.


Use &`operation_status` on the preceding camera `stop_source` operation ID after its media row finishes. Confirm the exact source is stopped before continuing with the separate microphone recording. If still active, stop here and retry this status question later.


Call &`start_microphone` for this recording path after I grant real microphone permission. Report its immediate operation ID/state; do not begin recording until the microphone source is confirmed.


Use &`operation_status` on the preceding recording-workflow `start_microphone` operation ID after I grant permission. Report its actual state and source ID only when completed; otherwise stop and ask me to retry later.


Use &`start_recording` on the confirmed microphone source ID, with `save_to=None` and a 60-second wall-clock ceiling. Report the new recording operation ID and immediate state. The media is in memory, with no automatic file save.


After at least three seconds of active recording, use &`pause_recording` on the exact recording operation ID above. Report the control operation ID/state, not a final clip.


Use &`operation_status` on the original recording operation ID. Confirm it is actually paused before the next step; if not, stop and ask me to retry this status question.


Use &`resume_recording` on that same verified paused recording operation ID. Report the immediate control operation ID/state; do not stop the recording yet.


Use &`operation_status` on the original recording operation ID and confirm it is running. I will then allow at least three more active seconds before the stop question. If it is not running, stop here.


After at least three more active seconds since resume, use &`stop_recording` on the original recording operation ID. Report the stop-control operation ID/state; the encoded clip belongs to the original recording receipt, not this control reply.


After the original recording media row completes, use &`operation_status` on the original `start_recording` operation ID. Report the recording duration reported by status, MIME type, byte count, and managed media ID if provided, without showing bytes. If still running, stop and retry later.


Use &`stop_source` on the exact microphone source ID from this recording workflow. Report the stop operation ID/state and do not stop another source.


Use &`operation_status` on the preceding recording-microphone `stop_source` operation ID. Confirm terminal stopped state before starting another microphone for the level reading. If it remains active, stop here and retry later.


Use &`start_microphone` after I grant real microphone permission. Report the immediate operation ID/state, without claiming a source ID until it completes.


Use &`operation_status` on the previous `start_microphone` operation ID after permission. Report the actual state and source ID only if completed; otherwise stop and retry later.


While I make a brief sound, use &`read_audio_levels` on the confirmed microphone source ID. Report the immediate operation ID/state, not an invented level.


Use &`operation_status` on the preceding `read_audio_levels` operation ID after it completes. Report the actual RMS level and units if present; if still running, stop and retry later.


Use &`stop_source` on the exact microphone source ID from this workflow. Report its immediate control operation ID/state.


Use &`operation_status` on the preceding microphone `stop_source` operation ID. Confirm terminal stopped state before the later one-shot recording; if not complete, stop and retry later.


Use &`record_camera` with `audio=False`, `duration=5`, and `save_to=None` while the real camera faces a neutral wall. Report only the immediate operation ID/state, then wait for permission and the five-second recording.


After the five-second wall-camera recording completes, use &`operation_status` on the preceding `record_camera` operation ID. Report actual duration, MIME type, byte count, and managed media ID if available; never include encoded bytes. If still running, stop and retry later.


Use &`record_microphone` with `duration=7` and `save_to=None` after I grant real microphone permission. Report only its immediate operation ID/state, then wait for the recording.


After the seven-second microphone recording completes, use &`operation_status` on the preceding `record_microphone` operation ID. Report actual duration, MIME type, byte count, and managed media ID if present; never include audio bytes. If still running, stop and retry later.


Use &`setup_share` to check actual screen-sharing support. Report its immediate operation ID/state and wait for the terminal result before choosing a screen; do not share anything silently.


Use &`operation_status` on the `setup_share` operation ID after it completes. Report the actual support state. If unsupported, stop the screen workflow and explain that result.


Use &`start_share` with `audio=False` only after I choose a neutral tab in the visible browser Share dialog. Report its immediate operation ID/state; do not claim a source ID yet.


After I choose the tab, use &`operation_status` on the `start_share` operation ID. Report the actual state and source ID only if completed; otherwise stop and retry later.


Use &`capture_screen` on the exact confirmed share source ID with `save_to=None`. Report its immediate operation ID/state, not image dimensions yet.


Use &`operation_status` on the `capture_screen` operation ID after completion. Report actual image dimensions and managed media ID if present, not pixels or private screen content.


Use the &`capture_tool` alias on that same confirmed share source ID with `save_to=None`. Report its immediate operation ID/state.


Use &`operation_status` on the `capture_tool` operation ID after completion. Report actual dimensions and managed media ID if present; do not include pixels.


Use &`stop_share` on the exact confirmed share source ID. Report its immediate stop operation ID/state.


Use &`operation_status` on the preceding `stop_share` operation ID. Report its actual terminal state and confirm the exact share source stopped before cleanup.


Use &`release_media` only for the exact managed media IDs reported by the completed AI still, recording, camera clip, microphone clip, screen, and capture_tool results above. Report each release operation ID/state. If an ID was not reported, skip it rather than guessing. Released browser bytes do not erase saved notebook displays.


After the release media rows finish, use &`operation_status` on each exact release operation ID from the prior answer. Report which managed captures were actually released and any failure; do not infer completion from an initial snapshot.
